# AKE Lab 5 · Optional: Reinforcement Learning mit DQN — Starter-Notebook

**Vertiefung zu Hause · zugleich die Live-Demo der Vorlesung** · Folien *Zeitreihen & Reinforcement Learning*, Abschnitt 6 ·
Lektüre: Sutton & Barto Kap. 1, 3, 6.5 · Mnih et al. (2015)

Ein Stab steht auf einem Wagen (**CartPole**). Der Agent schiebt den Wagen in jedem Schritt nach links oder rechts. Solange der Stab
steht, gibt es +1 Belohnung; die Episode endet, wenn er mehr als 12° kippt, der Wagen aus dem Bild fährt oder nach 500 Schritten.
Ein Deep Q-Network (DQN) lernt das Balancieren – nur aus Belohnung, ohne Physikmodell.

**Rechenzeit:** Training ≈ 2–4 Min. auf der CPU. Wer nicht trainieren will, setzt in Aufgabe R3 `TRAINIEREN = False`: Dann werden
die mitgelieferten Gewichte aus `results_referenz/dqn_cartpole.pt` geladen. Zwei kurze Lücken (`CODE`), sonst `AUSFÜHREN`.
**Erwartung:** Zufallspolitik ≈ 22 Schritte · gelöst (Mittel der letzten 20 Episoden ≥ 475) nach ≈ 430 Episoden · danach 500 Schritte.

In [ ]:
import time, json, random, collections
from pathlib import Path
import numpy as np, torch, torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(1)                          # ein Thread: kleines Netz, reproduzierbare Ergebnisse
env = gym.make("CartPole-v1"); env.action_space.seed(SEED)
RESULTS, REFDIR = Path("results"), Path("results_referenz"); RESULTS.mkdir(exist_ok=True)
print("gymnasium", gym.__version__, "| Zustand:", env.observation_space.shape, "| Aktionen:", env.action_space.n)

## R1 · Die Umgebung und eine Zufallspolitik

`AUSFÜHREN` · Der Zustand besteht aus vier Zahlen: Wagenposition, Wagengeschwindigkeit, Stabwinkel (rad), Winkelgeschwindigkeit.
Eine Politik, die zufällig links/rechts wählt, hält den Stab im Mittel nur gut 20 Schritte.

In [ ]:
def episode(politik, seed):
    """Spielt eine Episode mit politik(zustand) -> Aktion; gibt Return und Zustandsverlauf zurueck."""
    s, _ = env.reset(seed=seed); verlauf, ret, fertig = [s], 0.0, False
    while not fertig:
        s, r, term, trunc, _ = env.step(politik(s)); ret += r; verlauf.append(s); fertig = term or trunc
    return ret, np.array(verlauf)

zufall = lambda s: env.action_space.sample()
rets = [episode(zufall, 1000 + i)[0] for i in range(20)]
print(f"Zufallspolitik: mittlerer Return {np.mean(rets):.1f} (min {min(rets):.0f}, max {max(rets):.0f})")

def animiere(verlauf, titel):
    """Zeichnet Wagen und Stab aus dem Zustandsverlauf (ohne pygame) und gibt eine HTML-Animation zurueck."""
    fig, ax = plt.subplots(figsize=(6, 2.6)); ax.set_xlim(-2.6, 2.6); ax.set_ylim(-0.3, 1.4); ax.set_aspect("equal")
    ax.axhline(0, color="grey", lw=1); ax.set_yticks([]); ax.set_title(titel)
    wagen = plt.Rectangle((0, 0), 0.5, 0.25, color="#1F3864"); ax.add_patch(wagen)
    (stab,) = ax.plot([], [], lw=5, color="#E87722"); text = ax.text(-2.5, 1.25, "")
    def bild(i):
        x, _, th, _ = verlauf[i]
        wagen.set_xy((x - 0.25, 0)); stab.set_data([x, x + np.sin(th)], [0.25, 0.25 + np.cos(th)]); text.set_text(f"Schritt {i}")
        return wagen, stab, text
    anim = animation.FuncAnimation(fig, bild, frames=range(0, len(verlauf), 2), interval=40, blit=True); plt.close(fig)
    return HTML(anim.to_jshtml())

ret, verlauf = episode(zufall, 1000)
display(animiere(verlauf, f"Zufallspolitik: {ret:.0f} Schritte"))

## R2 · Das DQN

Ein Netz $Q(s, a; \theta)$ schätzt für jeden Zustand den Wert beider Aktionen (4 → 128 → 128 → 2, 17 410 Parameter). Gelernt wird mit
dem **TD-Ziel** aus Q-Learning, berechnet mit einem langsam nachgeführten **Zielnetz** $\theta^-$, auf zufälligen Minibatches aus dem
**Replay-Puffer**:

$$\mathcal L(\theta) = \Big(\underbrace{r + \gamma\,(1 - \text{ende}) \max_{a'} Q(s', a'; \theta^-)}_{\text{TD-Ziel}} - Q(s, a; \theta)\Big)^2$$

`CODE` (je 1 Zeile): **R2a** ε-greedy – mit Wahrscheinlichkeit ε eine Zufallsaktion, sonst die Aktion mit dem größten Q-Wert.
**R2b** das TD-Ziel für einen Minibatch (`R`, `T` = 1 bei Episodenende, `q_ziel(S2)` gibt die Q-Werte des Zielnetzes).

In [ ]:
GAMMA, LR, BATCH, PUFFER, TAU = 0.99, 1e-4, 128, 10_000, 0.005
EPS_START, EPS_ENDE, EPS_ZEIT = 0.9, 0.05, 1000      # epsilon faellt exponentiell von 0,9 auf 0,05
def netz(): return nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
q, q_ziel = netz(), netz(); q_ziel.load_state_dict(q.state_dict())
print("Parameter:", sum(p.numel() for p in q.parameters()))

def waehle_aktion(s, eps):
    # TODO R2a: mit Wahrscheinlichkeit eps env.action_space.sample(), sonst int(q(torch.tensor(s)).argmax())
    raise NotImplementedError("TODO R2a")

def td_ziel(R, S2, T):
    with torch.no_grad():
        # TODO R2b: R + GAMMA * (1 - T) * bester Q-Wert des Zielnetzes im Folgezustand
        raise NotImplementedError("TODO R2b")

# Prueffall: Belohnung 1, Folgezustand nicht terminal -> Ziel = 1 + 0,99 * max Q
S2 = torch.zeros(1, 4); print("TD-Ziel-Probe:", float(td_ziel(torch.ones(1), S2, torch.zeros(1))), "=", 1 + GAMMA * float(q_ziel(S2).max().detach()))

## R3 · Training

`AUSFÜHREN` · Bis zu 600 Episoden; Abbruch 10 Episoden, nachdem das Mittel der letzten 20 Episoden 475 erreicht hat.
RL-Lernkurven sind unruhig – Einbrüche zwischendurch sind normal.

In [ ]:
TRAINIEREN = True                                         # False: mitgelieferte Gewichte laden (≈ 2–4 Min. gespart)
if TRAINIEREN:
    opt = torch.optim.AdamW(q.parameters(), LR, amsgrad=True)
    puffer = collections.deque(maxlen=PUFFER); returns, schritte, geloest, t0 = [], 0, None, time.time()
    for ep in range(600):
        s, _ = env.reset(seed=SEED + ep); fertig, ret = False, 0.0
        while not fertig:
            eps = EPS_ENDE + (EPS_START - EPS_ENDE) * np.exp(-schritte / EPS_ZEIT)
            a = waehle_aktion(s, eps)
            s2, r, term, trunc, _ = env.step(a); fertig = term or trunc
            puffer.append((s, a, r, s2, float(term))); s = s2; ret += r; schritte += 1
            if len(puffer) >= BATCH:                      # Lernschritt auf einem zufaelligen Minibatch
                S, A, R, S2, T = map(lambda x: torch.tensor(np.array(x), dtype=torch.float32), zip(*random.sample(puffer, BATCH)))
                wert = q(S).gather(1, A.long().unsqueeze(1)).squeeze(1)       # Q(s, a) der ausgefuehrten Aktionen
                verlust = nn.functional.smooth_l1_loss(wert, td_ziel(R, S2, T))
                opt.zero_grad(); verlust.backward(); torch.nn.utils.clip_grad_value_(q.parameters(), 100); opt.step()
                with torch.no_grad():                     # Zielnetz weich nachfuehren
                    for pz, p in zip(q_ziel.parameters(), q.parameters()): pz.mul_(1 - TAU).add_(TAU * p)
        returns.append(ret); m20 = np.mean(returns[-20:])
        if ep % 50 == 0: print(f"Episode {ep:3d} | Schritte {schritte:6d} | Mittel(20) {m20:6.1f} | {time.time() - t0:5.0f} s")
        if geloest is None and len(returns) >= 20 and m20 >= 475: geloest = ep
        if geloest is not None and ep >= geloest + 10: break
    print(f"fertig: {len(returns)} Episoden, {schritte} Schritte, {time.time() - t0:.0f} s, geloest bei Episode {geloest}")
    torch.save(q.state_dict(), RESULTS / "dqn_cartpole.pt")
    json.dump({"returns": returns, "schritte": schritte, "geloest_ep": geloest}, open(RESULTS / "dqn_log.json", "w"))
else:
    q.load_state_dict(torch.load(REFDIR / "dqn_cartpole.pt")); returns = json.load(open(REFDIR / "dqn_log.json"))["returns"]

r = np.array(returns)
plt.plot(r, color="lightgrey", label="Return je Episode"); plt.plot(np.convolve(r, np.ones(20) / 20, "valid"), label="Mittel (20)")
plt.axhline(475, ls="--", color="orange"); plt.xlabel("Episode"); plt.ylabel("Return"); plt.legend(); plt.show()

## R4 · Die gelernte Politik

`AUSFÜHREN` · Greedy (ε = 0) auf 20 neuen Startzuständen. **Erwartung:** Return 500 – der Stab bleibt bis zum Ende stehen.

In [ ]:
greedy = lambda s: waehle_aktion(s, 0.0)
rets = [episode(greedy, 2000 + i)[0] for i in range(20)]
print(f"gelernte Politik: mittlerer Return {np.mean(rets):.1f} (min {min(rets):.0f})")
ret, verlauf = episode(greedy, 2000)
display(animiere(verlauf[:300], f"DQN: {ret:.0f} Schritte (gezeigt: die ersten 300)"))

**`DENKEN`**

a) Das Netz brauchte rund 50 000 Interaktionen. Was hieße das an einer echten Anlage – und was braucht man deshalb fast immer?
b) Warum ist die Lernkurve so unruhig, und wie berichtet man RL-Ergebnisse seriös?
c) Was passiert, wenn Sie in R2b das `(1 - T)` weglassen?